# Beta-t-EGARCH negative log-likelihood

This helper evaluates the objective used by the estimation notebook. It is a function, rather than a standalone analysis: call it through `f_s = @(x) Beta_univ_t_Egarch_Logl(x,Y,Inf_s,I_s)` and pass `f_s` to an optimizer.


## 0. Inputs and parameter conventions

`par` contains the location, long-run log scale, persistence, score loading, log degrees of freedom, and—when present—a leverage loading. `y` is a column vector of returns. `inf` selects information-standardized (`1`) or raw (`0`) scores; `I` selects the integrated (`1`) or stationary (`0`) scale recursion.

The degrees of freedom are parameterized as \(\nu=\exp(\texttt{vega\_l})\), which enforces positivity during optimization.


## 1. Filtering recursion

For each observation, the function computes the standardized residual \(e_t=(y_t-\mu)\exp(-\lambda_t)\), its Beta-type transform

$$B_t=\frac{e_t^2/\nu}{1+e_t^2/\nu},$$

and the Student's *t* score \(u_t=((\nu+1)B_t-1)/\mathcal I\). The next log scale follows either an integrated update or the mean-reverting AR(1) update, with an optional signed leverage contribution.


## 2. Objective returned to the optimizer

The last lines sum the conditional Student's *t* log densities and change the sign. Consequently, the function returns a **negative log-likelihood** suitable for minimization by `fmincon` or `fminsearch`.

## 3. Function implementation


In [ ]:
function Logl= Beta_univ_t_Egarch_Logl (par,y,inf,I)

if I==1
    
    mu = par(1); %mean
    omega = par(2); %unconstrained mean lambda scale
    kapa= par(3); %dinamic cond score par
    vega_l = par(4); %log shape parameter
    
    vega=exp(vega_l);

    if size(par,1)==5
        kapa_2=par(5);
    end
    
else
    
    mu = par(1); %mean
    omega = par(2); %unconstrained mean lambda scale
    phi = par(3); %dinamic AR parameter scale
    kapa= par(4); %dinamic cond score par
    vega_l = par(5); %log shape parameter

    vega=exp(vega_l);

    if size(par,1)==6
        kapa_2=par(6);
    end
    
end


T=size(y,1);

lam=zeros(T,1);
res=zeros(T,1);
u=zeros(T,1);
Beta=zeros(T,1);

if inf==1
    Inf=2*vega/(vega+3);
else
    Inf=1;
end

lam_=omega;

for i=1:T
   
	lam(i)=lam_;
	res(i)=(y(i)-mu)*exp(-lam(i));

    Beta(i)=((res(i)^2)/vega)/(1+((res(i)^2)/vega));
    u(i)=((vega+1)*Beta(i)-1)/Inf;
    
    if I==1
        
        if size(par,1)==5
            lam_=lam(i)+kapa*u(i)+kapa_2*sign(mu-y(i))*(u(i)+1);
        else
            lam_=lam(i)+kapa*u(i);
        end
   	
    else
        
        if size(par,1)==6
            lam_=omega*(1-phi)+phi*lam(i)+kapa*u(i)+kapa_2*sign(mu-y(i))*(u(i)+1);
        else
            lam_=omega*(1-phi)+phi*lam(i)+kapa*u(i);
        end

    end

end

LoglresSum=log(ones(T,1)+1/vega*res.^2);

Logl=T*log(gamma((vega+1)/2)/(gamma(vega/2)*sqrt(pi*vega)))-sum(lam)-(vega+1)/2*sum(LoglresSum);

Logl=-Logl;
